# Binary Signal generation

## All required imports

In [20]:
import pandas as pd
import os
import re
import numpy as np
from stockstats import wrap
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## Preprocessing

In [21]:
# Variables

col_names = ['timestamp', 'open', 'high', 'low', 'close', 'volume']


In [22]:
# Data extraction from Excel file
raw_df = pd.read_excel("../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/RECENT_DATA_FILE_DUMP_EURUSD_M1.xlsx", header = None)

In [27]:
# Preparing clean dataframe from raw dataframe
df = raw_df[0].str.split(";", expand=True)
df.columns = col_names

# Removing extra data
clean_df  = df.drop(columns=['volume'])

# Convert price columns to float - Type of data checked
clean_df [['open', 'high', 'low', 'close']] = clean_df[['open', 'high', 'low', 'close']].astype(float)
clean_df ['timestamp'] = pd.to_datetime(clean_df ['timestamp'], format="%Y%m%d %H%M%S")

#Creating index from timestamp
clean_df.set_index('timestamp', inplace=True)


#Make sure data has no duplicate values
print(clean_df.index.is_unique)
duplicated_data = clean_df.index[clean_df.index.duplicated()]
clean_df = clean_df[~clean_df.index.duplicated(keep='first')]
print(clean_df.head())


False
                        open     high      low    close
timestamp                                              
2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974
2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972


In [30]:
#Combine Text File generator
text_file_directory = "../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files/"

def get_dir_files(text_file_directory: str) -> list: 
    """
    Syntax: os.listdir(path)   
    Parameters: path (optional) :  path of the directory  
    Return Type: This method returns the list of all files and directories in the specified path. The return type of this method is list. 
    """
    all_files = os.listdir(text_file_directory)
    return all_files

def combine_text_files(all_files: list):
    print(f"{all_files}")
    with open(f'{text_file_directory}/combined_txt_file.txt', 'a') as file:
        for a_file in all_files:
            print(f"{a_file}")
            with open(f'{text_file_directory}/{a_file}', 'r') as temp_file:
                file.write(temp_file.read() + '\n')
    return f'{text_file_directory}/combined_txt_file.txt'

all_files = get_dir_files(text_file_directory)
combine_text_file_path = combine_text_files(all_files)

print(f"New file created: {combine_text_file_path}")

['DAT_ASCII_EURUSD_M1_2023.txt', 'DAT_ASCII_EURUSD_M1_2024.txt', 'DAT_ASCII_EURUSD_M1_202501.txt', 'DAT_ASCII_EURUSD_M1_202502.txt', 'DAT_ASCII_EURUSD_M1_202503.txt', 'DAT_ASCII_EURUSD_M1_202504.txt']
DAT_ASCII_EURUSD_M1_2023.txt
DAT_ASCII_EURUSD_M1_2024.txt
DAT_ASCII_EURUSD_M1_202501.txt
DAT_ASCII_EURUSD_M1_202502.txt
DAT_ASCII_EURUSD_M1_202503.txt
DAT_ASCII_EURUSD_M1_202504.txt
New file created: ../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files//combined_txt_file.txt


In [34]:
def parse_gap_report(file_path):
    gaps = []
    with open(file_path, 'r') as f:
        for line in f:
            match = re.match(r"Gap of (\d+)s found between (\d{14}) and (\d{14})\.", line)
            if match:
                duration = int(match.group(1))
                start = pd.to_datetime(match.group(2), format="%Y%m%d%H%M%S")
                end = pd.to_datetime(match.group(3), format="%Y%m%d%H%M%S")
                gaps.append({"start": start, "end": end, "duration_s": duration})
    return gaps

def handling_gaps(clean_df, combine_text_file_path):
    if combine_text_file_path:
        gaps = parse_gap_report(combine_text_file_path)
        # Forward fill small gaps (<= 300s) at 1-minute level
        clean_df = clean_df.asfreq('1T')
        # Flag large gaps (> 300s)
        for gap in gaps:
            #if gap['duration_s'] <= 300:
                #clean_df.loc[gap['start']:gap['end']] = clean_df.asfreq('1T', method='ffill')
            if gap['duration_s'] > 300:
                # Mark period as unreliable (e.g., set to NaN or flag)
                clean_df.loc[gap['start']:gap['end']] = None
                # missing_values = pd.DataFrame([gap['start'], gap['end']]) Have to improve in future
    else:
        # Forward fill all gaps if no gap report
        clean_df = clean_df.asfreq('1T', method='ffill')    
    return clean_df


#Execution of above funtions
clean_df = handling_gaps(clean_df, combine_text_file_path)
df_1min = clean_df.dropna()

### Labeling

In [35]:
# Labeling for recent 1m, 3m and 5m data at specific point in dataframe

df_1min = df_1min.copy()  

prev_close_1_min = df_1min['close'].shift(1)
conditions = [
    df_1min['close'] > prev_close_1_min,
    df_1min['close'] < prev_close_1_min
]
choices = [1, 2]

df_1min.loc[:, 'label_1min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_1min'] = 0


# 3 minutes candle
prev_close_3_min = df_1min['close'].shift(2)
conditions = [
    df_1min['close'] > prev_close_3_min,
    df_1min['close'] < prev_close_3_min
]
choices = [1, 2]

df_1min.loc[:, 'label_3min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_3min'] = 0



# 5 minutes candle
prev_close_3_min = df_1min['close'].shift(4)
conditions = [
    df_1min['close'] > prev_close_3_min,
    df_1min['close'] < prev_close_3_min 
]
choices = [1, 2]

df_1min.loc[:, 'label_5min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_5min'] = 0


# printing updated dataframe
df_1min.head(10)

,open,high,low,close,label_1min,label_3min,label_5min
timestamp,,,,,,,
2023-01-01 17:04:00,1.06970,1.06974,1.06970,1.06970,0,0,0
2023-01-01 17:05:00,1.06973,1.06978,1.06970,1.06971,1,0,0
2023-01-01 17:06:00,1.06966,1.06966,1.06966,1.06966,2,2,0
2023-01-01 17:08:00,1.06970,1.06974,1.06970,1.06974,1,1,0
2023-01-01 17:10:00,1.06975,1.06980,1.06972,1.06972,2,1,1
2023-01-01 17:11:00,1.06972,1.06972,1.06972,1.06972,0,2,1
2023-01-01 17:12:00,1.06975,1.06980,1.06975,1.06980,1,1,1
2023-01-01 17:13:00,1.07066,1.07066,1.06917,1.06943,2,2,2
2023-01-01 17:14:00,1.06937,1.06937,1.06899,1.06899,2,2,2


### Indicators

In [38]:
df_1min = df_1min.reset_index()  # stockstats requires 'timestamp' as a column

# Wrap with stockstats
sdf = wrap(df_1min)

# Compute indicators
sdf['close_5_ema']     # EMA(5)
sdf['close_10_ema']    # EMA(10)
sdf['rsi_14']          # RSI(14)
sdf['macdh']           # MACD Histogram
sdf['adx']             # ADX
sdf['atr']             # ATR
sdf['boll_ub']         # Bollinger Upper Band
sdf['boll_lb']         # Bollinger Lower Band
sdf['boll_width'] = sdf['boll_ub'] - sdf['boll_lb']  # Bollinger Band Width

# Manually compute Candle Body/Wick Ratio
sdf['body'] = abs(sdf['close'] - sdf['open'])
sdf['wick'] = sdf['high'] - sdf['low']
sdf['body_wick_ratio'] = sdf['body'] / sdf['wick'].replace(0, 1e-9)

# Optional: drop intermediate body/wick columns if you don't want them
# sdf.drop(columns=['body', 'wick'], inplace=True)


final_df = sdf[[
    'timestamp', 'open', 'high', 'low', 'close',
    'close_5_ema', 'close_10_ema', 'rsi_14', 'macdh',
    'adx', 'atr', 'boll_width', 'body_wick_ratio', 'label_1min', 'label_3min', 'label_5min'
]]

print(final_df.head(10))

            timestamp     open     high      low    close  close_5_ema  \
0 2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970     1.069700   
1 2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971     1.069706   
2 2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966     1.069684   
3 2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974     1.069707   
4 2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972     1.069712   
5 2023-01-01 17:11:00  1.06972  1.06972  1.06972  1.06972     1.069715   
6 2023-01-01 17:12:00  1.06975  1.06980  1.06975  1.06980     1.069745   
7 2023-01-01 17:13:00  1.07066  1.07066  1.06917  1.06943     1.069636   
8 2023-01-01 17:14:00  1.06937  1.06937  1.06899  1.06899     1.069415   
9 2023-01-01 17:15:00  1.06788  1.06788  1.06788  1.06788     1.068894   

   close_10_ema      rsi_14         macdh         adx       atr  boll_width  \
0      1.069700         NaN  0.000000e+00         NaN  0.000040         NaN   
1      1.069705  100.000000

### Defining training variables and target variables 

# Reruired output format gien by client
{
  "pair": "EUR/USD",
  "timeframe": "3m",
  "signal": "UP",
  "confidence": 0.734
}

Queries: 
What are the target variables should we aim for?
Should we only use trend as a target for all 1 min, 2 min and 3 min?